# Grammar Scoring Engine for Spoken Audio

**Task:** given a 45-60 second spoken-English recording, predict a continuous **MOS Likert grammar score between 0 and 5**.
**Data:** 769 labelled training clips, 216 test clips. **Metrics:** RMSE (lower is better) and Pearson correlation (higher is better).

### Approach in one paragraph
Grammar is a *linguistic* property, not an acoustic one ("I go to market yesterday" and "I went to the market yesterday" sound almost identical). So the model has to see **what was said**, not only how it sounded. The pipeline therefore fuses three views of every recording:

1. **Acoustic view**: MFCC, energy, ZCR, spectral, chroma and pause statistics from the waveform.
2. **Linguistic-statistics view**: hand-crafted features computed on a **Whisper ASR** transcript (length, sentence structure, lexical diversity, function words, hesitations, self-corrections).
3. **Language-representation view**: a **frozen pretrained sentence-transformer** embedding of the transcript.

With only 769 labelled clips, I deliberately avoid training a large network end to end. Pretrained encoders supply the representations, and small, well-regularised regressors (Ridge, ExtraTrees, Random Forest, HistGradientBoosting) learn the final mapping. Everything is judged with **5-fold out-of-fold (OOF) cross-validation**, and the final ensemble is evaluated on the training set (the compulsory training RMSE) as well.

```
audio.wav
   |-- Acoustic features (MFCC / energy / ZCR / spectral / chroma / pauses) ----------.
   |                                                                                   |
   '-- Whisper ASR --> transcript --+--> Linguistic features ---------------------->  Feature fusion
                                    '--> Frozen sentence embeddings ---------------->      |
                                                                                    Regression models
                                                                                           |
                                                                                    OOF validation + ensemble
                                                                                           |
                                                                                    grammar score (0-5)
```

### Notebook map
| # | Section |
|---|---|
| 1 | Setup, configuration and data loading |
| 2 | Exploratory analysis of labels and audio |
| 3 | Acoustic feature extraction |
| 4 | Speech recognition with Whisper |
| 5 | Linguistic features and sentence embeddings |
| 6 | Feature fusion |
| 7 | Cross-validated model comparison and ablation |
| 8 | Ensemble, final training and **training RMSE** |
| 9 | Visualisations and error analysis |
| 10 | Test prediction and submission file |
| 11 | Report |

> **Runtime notes (CPU-friendly defaults).** Whisper transcription is the slow step. The notebook defaults to `openai/whisper-base`, which is practical on a CPU-only machine (expect on the order of an hour or two for ~1000 clips; `openai/whisper-tiny` is faster still, `-small` / `-medium` are more accurate but need a GPU to be comfortable). Every expensive step caches its output in the working directory, so after the first run, re-running the notebook is cheap. Whisper and the sentence-transformer weights normally come from the Hugging Face Hub. **If your notebook has no internet** (Kaggle's default; you will see `Temporary failure in name resolution`), follow the *Offline setup* box in section 4: the notebook can load both models from an attached dataset, or reuse cached transcripts and embeddings. Set `CFG["debug_n"]` to a small number (e.g. 40) for a quick smoke test before committing to the full run.

## 1. Setup and configuration

In [ ]:
# Uncomment on a fresh Kaggle / Colab session if packages are missing
# !pip install -q librosa soundfile transformers sentence-transformers accelerate

In [ ]:
import os, re, glob, json, time, warnings, random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
import soundfile as sf
from joblib import Parallel, delayed
from tqdm.auto import tqdm
from scipy.stats import pearsonr
from scipy.optimize import nnls

from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "axes.titleweight": "bold"})

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Central configuration: change things here, not deep inside the notebook
CFG = {
    "sr": 16000,                                    # all audio is resampled to 16 kHz (Whisper's native rate)
    "n_mfcc": 20,
    "n_folds": 5,
    "whisper_model": "openai/whisper-base",         # CPU-friendly default; "-tiny" is faster, "-small"/"-medium" are more accurate (GPU advised)
    "embed_model": "sentence-transformers/all-MiniLM-L6-v2",
    "whisper_path": None,                           # optional: folder with a downloaded Whisper model (offline use)
    "embed_path": None,                             # optional: folder with a downloaded sentence-transformer (offline use)
    "allow_acoustic_only": True,                    # if no text models can be loaded, continue with acoustic features only (clearly flagged, much weaker)
    "n_jobs": -1,
    "debug_n": None,                                # e.g. 60 for a quick smoke test on a subset of train
}
print(CFG)

### 1.1 Locate the data
The notebook searches the usual locations (`/kaggle/input`, the current folder, `./data`) for `train.csv`. Audio files are matched by file name wherever they live under that folder, so the exact sub-folder layout does not matter.

In [ ]:
def find_data_dir():
    # Return the folder that contains train.csv (searches common locations)
    for root in [Path("/kaggle/input"), Path("."), Path("data"), Path("..")]:
        if root.exists():
            hits = sorted(root.rglob("train.csv"))
            if hits:
                return hits[0].parent
    raise FileNotFoundError("train.csv not found. Set DATA_DIR manually.")

DATA_DIR = find_data_dir()
WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
WORK_DIR.mkdir(parents=True, exist_ok=True)
print("DATA_DIR:", DATA_DIR)
print("WORK_DIR:", WORK_DIR)

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")

print("\ntrain:", train.shape, "| test:", test.shape, "| sample_submission:", sample_sub.shape)
display(train.head())
display(sample_sub.head())

In [ ]:
# Detect the file-name and label columns instead of hard-coding them
FILE_COL = next((c for c in train.columns if re.search(r"file|name|audio|path", c, re.I)), train.columns[0])
num_cols = [c for c in train.select_dtypes("number").columns if c != FILE_COL]
LABEL_COL = next((c for c in num_cols if re.search(r"label|score|grammar", c, re.I)), num_cols[-1])
TEST_FILE_COL = FILE_COL if FILE_COL in test.columns else test.columns[0]
SUB_FILE_COL, SUB_LABEL_COL = sample_sub.columns[0], sample_sub.columns[-1]
print(f"file column = {FILE_COL!r} | label column = {LABEL_COL!r} | submission columns = {list(sample_sub.columns)}")

# Map every file name in the csv files to a real .wav path
wav_files = list(DATA_DIR.rglob("*.wav"))
by_name = {p.name: p for p in wav_files}
by_stem = {p.stem: p for p in wav_files}
print(f"found {len(wav_files)} .wav files under {DATA_DIR}")

def resolve_path(fn):
    fn = str(fn)
    for key in (fn, Path(fn).name, Path(fn).stem):
        if key in by_name:
            return str(by_name[key])
        if key in by_stem:
            return str(by_stem[key])
    return None

train["path"] = train[FILE_COL].map(resolve_path)
test["path"] = test[TEST_FILE_COL].map(resolve_path)
print("missing audio -> train:", train.path.isna().sum(), "| test:", test.path.isna().sum())
assert train.path.notna().all() and test.path.notna().all(), "Some audio files could not be located"

if CFG["debug_n"]:
    train = train.sample(CFG["debug_n"], random_state=SEED).reset_index(drop=True)
    test = test.head(max(10, CFG["debug_n"] // 3)).reset_index(drop=True)
    print("DEBUG MODE: using", len(train), "train and", len(test), "test clips")

y_train = train[LABEL_COL].astype(float).values
n_train = len(train)

# One table with every clip (train first, then test) so features line up row by row
all_df = pd.concat([
    train[[FILE_COL, "path"]].rename(columns={FILE_COL: "filename"}).assign(split="train"),
    test[[TEST_FILE_COL, "path"]].rename(columns={TEST_FILE_COL: "filename"}).assign(split="test"),
], ignore_index=True)
is_train = (all_df.split == "train").values
print(all_df.split.value_counts().to_dict())

## 2. Exploratory analysis

A model cannot compensate for a dataset that has not been understood, so the first step is to look at the target and at the raw audio.

### 2.1 Target distribution

In [ ]:
print(train[LABEL_COL].describe().round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(y_train, bins=20, kde=True, ax=axes[0], color="#0B7A75")
axes[0].set(title="Grammar score distribution (train)", xlabel="grammar score", ylabel="clips")

rounded = pd.Series(np.round(y_train * 2) / 2).value_counts().sort_index()
axes[1].bar(rounded.index.astype(str), rounded.values, color="#A85A0C")
axes[1].set(title="Clips per score (rounded to 0.5)", xlabel="grammar score", ylabel="clips")
plt.tight_layout(); plt.show()

**Reading the plot.** The score range and balance matter: a model trained on skewed targets tends to regress towards the mean, which hurts RMSE on the extremes and shrinks the spread of predictions. This is kept in mind when judging the prediction distribution later.

### 2.2 Audio properties
For every clip: duration, sample rate, channels and frame count.

In [ ]:
def audio_info(path):
    i = sf.info(path)
    return {"duration": i.duration, "sample_rate": i.samplerate, "channels": i.channels, "frames": i.frames}

info_df = pd.DataFrame(Parallel(n_jobs=CFG["n_jobs"])(delayed(audio_info)(p) for p in tqdm(all_df.path)))
all_df = pd.concat([all_df, info_df], axis=1)

display(all_df.groupby("split")[["duration", "sample_rate", "channels"]].describe().T.round(2))
print("sample rates:", all_df.sample_rate.value_counts().to_dict())
print("channels    :", all_df.channels.value_counts().to_dict())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(data=all_df, x="duration", hue="split", bins=30, ax=axes[0], palette=["#0B7A75", "#A85A0C"])
axes[0].set(title="Audio duration by split", xlabel="seconds")

d_train = all_df.loc[is_train, "duration"].values
axes[1].scatter(d_train, y_train, alpha=0.5, s=18, color="#0F1B33")
r_dur = pearsonr(d_train, y_train)[0]
axes[1].set(title=f"Duration vs grammar score (r = {r_dur:.2f})", xlabel="duration (s)", ylabel="grammar score")
plt.tight_layout(); plt.show()

### 2.3 Waveform and Mel spectrogram: lowest vs highest scored clip

In [ ]:
tr_view = all_df[is_train].copy()
tr_view["label"] = y_train
examples = [("Lowest score", tr_view.loc[tr_view.label.idxmin()]),
            ("Highest score", tr_view.loc[tr_view.label.idxmax()])]

fig, axes = plt.subplots(2, 2, figsize=(14, 7))
for col, (title, row) in enumerate(examples):
    y_audio, sr_ = librosa.load(row.path, sr=CFG["sr"])
    librosa.display.waveshow(y_audio, sr=sr_, ax=axes[0, col], color="#0B7A75")
    axes[0, col].set_title(f"{title} (score {row.label:.2f}): waveform")
    mel = librosa.power_to_db(librosa.feature.melspectrogram(y=y_audio, sr=sr_, n_mels=64), ref=np.max)
    librosa.display.specshow(mel, sr=sr_, x_axis="time", y_axis="mel", ax=axes[1, col], cmap="magma")
    axes[1, col].set_title("Mel spectrogram")
plt.tight_layout(); plt.show()

The two spectrograms look broadly alike, which supports the central design decision: the acoustic signal alone says little about grammar, so a transcript-based view is needed.

## 3. Acoustic features

Classical speech features, summarised with simple statistics so each clip becomes one fixed-length vector:

| Feature | What it captures |
|---|---|
| MFCC (+ deltas) | spectral envelope of the voice and how it changes |
| RMS energy | loudness / intensity variation |
| Zero-crossing rate | noisiness and signal changes |
| Spectral centroid, bandwidth, roll-off, contrast | brightness and spectral shape |
| Chroma | pitch-class energy (weak for speech, cheap to include) |
| Pause statistics | voiced ratio, number of segments, gap lengths (a *fluency* proxy) |

Leading and trailing silence is trimmed first. Pause statistics are intentionally included, but they describe **fluency, not grammar**; the ablation in section 7 shows how much they really help.

In [ ]:
def summarize(prefix, M, feats):
    # Add mean / std / min / max of every row of a (n_features x frames) matrix
    M = np.atleast_2d(M)
    for i, row in enumerate(M):
        tag = f"{prefix}{i}" if M.shape[0] > 1 else prefix
        feats[f"{tag}_mean"] = float(row.mean())
        feats[f"{tag}_std"] = float(row.std())
        feats[f"{tag}_min"] = float(row.min())
        feats[f"{tag}_max"] = float(row.max())

def acoustic_features(path, sr=CFG["sr"], n_mfcc=CFG["n_mfcc"], top_db=30):
    feats = {}
    try:
        y_audio, _ = librosa.load(path, sr=sr, mono=True)
        y_t, _ = librosa.effects.trim(y_audio, top_db=top_db)
        if len(y_t) < sr:                       # trimming removed almost everything: keep the original
            y_t = y_audio
        dur = len(y_t) / sr
        feats["ac_speech_span_sec"] = dur

        mfcc = librosa.feature.mfcc(y=y_t, sr=sr, n_mfcc=n_mfcc)
        summarize("ac_mfcc", mfcc, feats)
        summarize("ac_dmfcc", librosa.feature.delta(mfcc), feats)
        summarize("ac_rms", librosa.feature.rms(y=y_t), feats)
        summarize("ac_zcr", librosa.feature.zero_crossing_rate(y_t), feats)
        summarize("ac_centroid", librosa.feature.spectral_centroid(y=y_t, sr=sr), feats)
        summarize("ac_bandwidth", librosa.feature.spectral_bandwidth(y=y_t, sr=sr), feats)
        summarize("ac_rolloff", librosa.feature.spectral_rolloff(y=y_t, sr=sr), feats)
        summarize("ac_contrast", librosa.feature.spectral_contrast(y=y_t, sr=sr), feats)
        summarize("ac_chroma", librosa.feature.chroma_stft(y=y_t, sr=sr), feats)

        # Pause / voicing statistics from energy-based segmentation
        intervals = librosa.effects.split(y_t, top_db=top_db)
        voiced = sum(e - s for s, e in intervals) / sr
        gaps = [(intervals[i + 1][0] - intervals[i][1]) / sr for i in range(len(intervals) - 1)]
        feats["ac_voiced_ratio"] = voiced / dur
        feats["ac_pause_ratio"] = 1 - voiced / dur
        feats["ac_n_segments"] = len(intervals)
        feats["ac_segments_per_sec"] = len(intervals) / dur
        feats["ac_mean_gap"] = float(np.mean(gaps)) if gaps else 0.0
        feats["ac_max_gap"] = float(np.max(gaps)) if gaps else 0.0
        feats["ac_n_long_pauses"] = int(sum(g > 0.5 for g in gaps))
    except Exception as exc:
        print(f"[warn] acoustic extraction failed for {path}: {exc}")
    return feats

AC_CACHE = WORK_DIR / "acoustic_features.csv"
if AC_CACHE.exists() and not CFG["debug_n"]:
    ac_df = pd.read_csv(AC_CACHE)
    print("loaded cached acoustic features", ac_df.shape)
else:
    t0 = time.time()
    rows = Parallel(n_jobs=CFG["n_jobs"])(delayed(acoustic_features)(p) for p in tqdm(all_df.path))
    ac_df = pd.DataFrame(rows)
    if not CFG["debug_n"]:
        ac_df.to_csv(AC_CACHE, index=False)
    print(f"extracted {ac_df.shape[1]} acoustic features in {time.time() - t0:.0f}s")
ac_df.head()

## 4. Speech recognition with Whisper

The problem changes from **Audio -> Grammar** into **Audio -> Language -> Grammar** while the acoustic information is still kept.
Whisper processes audio in 30-second windows, so the 45-60 second clips are chunked by the pipeline.

**A caveat worth knowing:** Whisper is trained on huge amounts of text and has a strong language-model prior, so it sometimes *silently corrects* a speaker's grammar mistakes. That weakens the signal in the transcript for the weakest speakers. The acoustic path and the error analysis in section 9 partly guard against this.

> ### Offline setup (no internet in the notebook)
> The Whisper and sentence-transformer weights have to come from somewhere. The next cells try, in this order: **(1)** cached files (`transcripts.csv`, `text_embeddings.npy`) in the working folder or in an attached dataset, **(2)** a model folder in an attached dataset or one set in `CFG["whisper_path"]` / `CFG["embed_path"]`, **(3)** the Hugging Face Hub if the internet is on. **(4)** if none of that is possible and `CFG["allow_acoustic_only"]` is `True`, the notebook still runs to the end using acoustic features only. It prints a loud warning, and the result is much weaker because grammar lives in the text.
>
> **Option A: turn internet on** (Kaggle: *Notebook options / Settings panel -> Internet -> On*; Kaggle may ask for phone verification). Note that some competitions require internet to be *off* for the final submission run, so prefer B or C for that.
>
> **Option B: attach the models as a dataset.** On any machine with internet (e.g. a free Colab session), run:
> ```python
> from huggingface_hub import snapshot_download
> snapshot_download("openai/whisper-base", local_dir="whisper-base")
> snapshot_download("sentence-transformers/all-MiniLM-L6-v2", local_dir="all-MiniLM-L6-v2")
> ```
> Upload both folders as one Kaggle dataset and **Add Input** it to this notebook. The notebook finds them automatically (folder names must contain `whisper-base` and `minilm`).
>
> **Option C: reuse cached outputs.** Run this notebook once anywhere with internet (a free Colab GPU makes Whisper fast), then upload the generated `transcripts.csv` and `text_embeddings.npy` as a dataset and attach it. Cached files are only used if they match this run's files row for row.

In [ ]:
import socket
try:
    import torch
    from transformers import pipeline
    HAS_TRANSFORMERS = True
except ImportError as exc:
    torch, HAS_TRANSFORMERS = None, False
    print("torch / transformers not available:", exc)

def hub_reachable(host="huggingface.co", timeout=5):
    # True if the Hugging Face Hub can be reached from this machine
    try:
        socket.create_connection((host, 443), timeout=timeout).close()
        return True
    except OSError:
        return False

def find_cached(name):
    # Look for a cache file in the working folder first, then in attached Kaggle inputs
    candidates = [WORK_DIR / name]
    if Path("/kaggle/input").exists():
        candidates += sorted(Path("/kaggle/input").rglob(name))
    return next((p for p in candidates if p.exists()), None)

def find_local_model(keywords, marker="config.json"):
    # Look for a model folder among attached Kaggle inputs whose path contains every keyword
    root = Path("/kaggle/input")
    if root.exists():
        for f in sorted(root.rglob(marker)):
            if all(k in str(f.parent).lower() for k in keywords):
                return str(f.parent)
    return None

def resolve_model(hub_id, local_keywords, override=None, marker="config.json"):
    # Priority: explicit path > attached dataset > Hugging Face Hub > a clear error message
    if override:
        return override
    local = find_local_model(local_keywords, marker)
    if local:
        print("using local model folder:", local)
        return local
    if hub_reachable():
        return hub_id
    raise RuntimeError(
        f"Cannot load '{hub_id}': no internet and no local copy found.\n"
        "Fix: (A) turn Internet ON in the notebook settings, or (B) attach a dataset that contains the downloaded "
        "model folder (see the 'Offline setup' box above), or (C) attach cached transcripts.csv / text_embeddings.npy."
    )

# 1) Cached transcripts (only accepted if they line up exactly with this run's files)
transcripts = None
tr_file = None if CFG["debug_n"] else find_cached("transcripts.csv")
if tr_file is not None:
    cached = pd.read_csv(tr_file)
    if len(cached) == len(all_df) and (cached["filename"].astype(str).values == all_df["filename"].astype(str).values).all():
        transcripts = cached["transcript"].fillna("").tolist()
        print("loaded cached transcripts from", tr_file)
    else:
        print("found", tr_file, "but it does not match this run's files; ignoring it")

# 2) Otherwise run Whisper
TEXT_AVAILABLE = True
if transcripts is None:
    try:
        if not HAS_TRANSFORMERS:
            raise RuntimeError("torch / transformers are not installed (pip install transformers torch).")
        whisper_src = resolve_model(CFG["whisper_model"], ["whisper", CFG["whisper_model"].split("-")[-1]],
                                    override=CFG.get("whisper_path"))
    except RuntimeError as exc:
        if not CFG.get("allow_acoustic_only", False):
            raise
        TEXT_AVAILABLE = False
        print("=" * 72)
        print("WARNING: Whisper is not available ->", exc)
        print("Continuing in ACOUSTIC-ONLY mode. Results will be much weaker, because grammar")
        print("lives in the text. Fix the model access and re-run from this cell to get the full pipeline.")
        print("=" * 72)

    if TEXT_AVAILABLE:
        device = 0 if torch.cuda.is_available() else -1
        print("device:", "GPU" if device == 0 else "CPU", "| model:", whisper_src)
        asr = pipeline(
            "automatic-speech-recognition",
            model=whisper_src,
            device=device,
            chunk_length_s=30,
            torch_dtype=torch.float16 if device == 0 else torch.float32,
        )

        def transcribe_all(paths, batch=16):
            out = []
            for i in tqdm(range(0, len(paths), batch), desc="Whisper"):
                audio = [{"raw": librosa.load(p, sr=CFG["sr"], mono=True)[0], "sampling_rate": CFG["sr"]}
                         for p in paths[i:i + batch]]
                res = asr(audio, batch_size=batch, generate_kwargs={"language": "en", "task": "transcribe"})
                out += [r["text"].strip() for r in res]
            return out

        transcripts = transcribe_all(all_df.path.tolist())
        if not CFG["debug_n"]:
            pd.DataFrame({"filename": all_df.filename, "transcript": transcripts}).to_csv(WORK_DIR / "transcripts.csv", index=False)
        del asr
        if device == 0:
            torch.cuda.empty_cache()
    else:
        transcripts = [""] * len(all_df)

all_df["transcript"] = pd.Series(transcripts).fillna("").values
print("empty transcripts:", (all_df.transcript.str.len() == 0).sum())

In [ ]:
# A few transcripts next to their human scores
if TEXT_AVAILABLE:
    sample_view = all_df[is_train].assign(score=y_train).sort_values("score")
    mid = len(sample_view) // 2
    pick = pd.concat([sample_view.head(3), sample_view.iloc[max(mid - 1, 0): mid + 2], sample_view.tail(3)])
    for _, r in pick.iterrows():
        print(f"[score {r['score']:.2f}] {r['transcript'][:350]}\n")
else:
    print("Transcripts are not available in this run (acoustic-only mode).")

## 5. Linguistic features and sentence embeddings

### 5.1 Hand-crafted linguistic features
These are **signals, not a grammar checker**. Each captures one thing a human rater implicitly notices:

* **Length and sentence structure**: word count, sentence count, mean / max / std sentence length.
* **Lexical diversity and repetition**: type-token ratio, root TTR, adjacent-word repeats, repeated bigrams.
* **Function words**: articles, auxiliaries and prepositions (`the, a, an, is, are, was, were, have, has, had ...`), whose presence or absence tracks grammatical completeness.
* **Subordination and connectives**: `because, although, which, when ...` hint at complex sentences.
* **Hesitations and self-corrections**: `um, uh, erm` and `actually, sorry, I mean, rather, instead`; the rubric explicitly rewards self-correction.

In [ ]:
FUNCTION_WORDS = ["the", "a", "an", "is", "are", "was", "were", "have", "has", "had", "be", "been",
                  "do", "does", "did", "to", "of", "in", "on", "at", "for", "with"]
KEY_FUNCTION_WORDS = ["the", "a", "an", "is", "are", "was", "were", "have", "has", "had"]
CONNECTIVES = ["because", "although", "though", "which", "who", "that", "when", "while", "since",
               "if", "but", "and", "so", "however", "therefore", "whereas", "unless"]
SUBORDINATORS = ["because", "although", "though", "which", "who", "while", "since", "unless", "whereas", "if", "when"]
HESITATIONS = {"um", "uh", "erm", "uhm", "hmm", "er", "ah", "mm"}
SELF_CORRECTION_PATTERNS = [r"\bactually\b", r"\bsorry\b", r"\bi mean\b", r"\brather\b", r"\binstead\b"]

def linguistic_features(text):
    text = text if isinstance(text, str) else ""
    words = re.findall(r"[a-z']+", text.lower())
    n = len(words)
    n_safe = max(n, 1)
    sents = [s for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s]
    sent_len = np.array([len(re.findall(r"[a-z']+", s.lower())) for s in sents]) if sents else np.array([0])
    counts = pd.Series(words).value_counts() if n else pd.Series(dtype=int)

    f = {}
    f["ling_n_words"] = n
    f["ling_n_sents"] = len(sents)
    f["ling_avg_sent_len"] = float(sent_len.mean())
    f["ling_max_sent_len"] = float(sent_len.max())
    f["ling_std_sent_len"] = float(sent_len.std())
    f["ling_avg_word_len"] = float(np.mean([len(w) for w in words])) if n else 0.0
    f["ling_long_word_ratio"] = sum(len(w) >= 7 for w in words) / n_safe
    f["ling_commas_per_sent"] = text.count(",") / max(len(sents), 1)

    uniq = len(set(words))
    f["ling_n_unique"] = uniq
    f["ling_ttr"] = uniq / n_safe
    f["ling_root_ttr"] = uniq / np.sqrt(n_safe)
    f["ling_adjacent_repeats"] = sum(a == b for a, b in zip(words, words[1:])) / n_safe
    bigrams = list(zip(words, words[1:]))
    f["ling_bigram_repeat"] = 1 - len(set(bigrams)) / max(len(bigrams), 1)
    f["ling_top_word_share"] = float(counts.iloc[0] / n_safe) if n else 0.0

    f["ling_func_ratio"] = sum(w in FUNCTION_WORDS for w in words) / n_safe
    for w in KEY_FUNCTION_WORDS:
        f[f"ling_fw_{w}"] = words.count(w) / n_safe
    f["ling_connective_ratio"] = sum(w in CONNECTIVES for w in words) / n_safe
    f["ling_subordinator_count"] = sum(w in SUBORDINATORS for w in words)
    f["ling_pronoun_i_ratio"] = words.count("i") / n_safe

    n_hes = sum(w in HESITATIONS for w in words)
    f["ling_hesitation_count"] = n_hes
    f["ling_hesitation_ratio"] = n_hes / n_safe
    n_sc = sum(len(re.findall(p, text.lower())) for p in SELF_CORRECTION_PATTERNS)
    f["ling_selfcorrection_count"] = n_sc
    f["ling_selfcorrection_ratio"] = n_sc / n_safe
    return f

ling_df = pd.DataFrame([linguistic_features(t) for t in all_df.transcript])

# Cross-modal fluency features: words per second of audio and per second of actual voicing
span = ac_df["ac_speech_span_sec"].replace(0, np.nan)
ling_df["ling_words_per_sec"] = ling_df["ling_n_words"] / span
ling_df["ling_articulation_rate"] = ling_df["ling_n_words"] / (ac_df["ac_voiced_ratio"] * span).replace(0, np.nan)
print("linguistic features:", ling_df.shape[1])
ling_df.describe().T.head(12).round(3)

### 5.2 Frozen sentence-transformer embeddings
Hand-crafted counts cannot capture a sentence like *"Although I was tired, I decided to finish the project because the deadline was approaching."* A pretrained encoder turns the transcript into a dense vector that carries semantic and syntactic information learned from large text corpora.

The encoder is **frozen**: with only a few hundred labelled examples, fine-tuning a transformer is mostly an overfitting exercise.

In [ ]:
emb = None
EMB_AVAILABLE = TEXT_AVAILABLE
emb_file = None if (CFG["debug_n"] or not TEXT_AVAILABLE) else find_cached("text_embeddings.npy")
if emb_file is not None:
    cached_emb = np.load(emb_file)
    if cached_emb.shape[0] == len(all_df):
        emb = cached_emb
        print("loaded cached embeddings from", emb_file, emb.shape)
    else:
        print("found", emb_file, "but its row count does not match this run; ignoring it")

if emb is None and TEXT_AVAILABLE:
    try:
        from sentence_transformers import SentenceTransformer
        embed_src = resolve_model(CFG["embed_model"], ["minilm"], override=CFG.get("embed_path"), marker="modules.json")
        use_cuda = torch is not None and torch.cuda.is_available()
        st_model = SentenceTransformer(embed_src, device="cuda" if use_cuda else "cpu")
        emb = st_model.encode(all_df.transcript.tolist(), batch_size=64, show_progress_bar=True, normalize_embeddings=True)
        if not CFG["debug_n"]:
            np.save(WORK_DIR / "text_embeddings.npy", emb)
        del st_model
        print("embeddings:", emb.shape)
    except (RuntimeError, ImportError) as exc:
        if not CFG.get("allow_acoustic_only", False):
            raise
        EMB_AVAILABLE = False
        print("WARNING: sentence embeddings are not available ->", exc)
        print("Continuing without the embedding block.")

if emb is None:
    emb_df = pd.DataFrame(index=range(len(all_df)))          # empty block
else:
    emb_df = pd.DataFrame(emb, columns=[f"emb_{i}" for i in range(emb.shape[1])])
print("embedding block shape:", emb_df.shape)

## 6. Feature fusion

The three views are concatenated into one table. Missing or infinite values are imputed with **training-set medians only** (so nothing leaks from the test set), and zero-variance columns are dropped. The block each column came from is recorded, which makes the ablation and importance analyses below possible.

In [ ]:
blocks_raw = {"acoustic": ac_df, "linguistic": ling_df, "embeddings": emb_df}
X_all = pd.concat(blocks_raw.values(), axis=1).replace([np.inf, -np.inf], np.nan)
X_all = X_all.fillna(X_all[is_train].median()).fillna(0.0)

# Drop columns that never vary on the training set
keep = X_all[is_train].std() > 1e-9
X_all = X_all.loc[:, keep]

block_of = {c: name for name, df in blocks_raw.items() for c in df.columns if c in X_all.columns}
blocks = {name: [c for c in X_all.columns if block_of[c] == name] for name in blocks_raw}

X_tr = X_all[is_train].reset_index(drop=True)
X_te = X_all[~is_train].reset_index(drop=True)
print(f"fused features: {X_all.shape[1]}  ->  " + ", ".join(f"{k}: {len(v)}" for k, v in blocks.items()))
print("train matrix:", X_tr.shape, "| test matrix:", X_te.shape)

## 7. Model comparison with 5-fold cross-validation

**Why OOF validation?** A flexible model can fit its own training data almost perfectly and still fail on new audio. Each training clip therefore receives exactly one **out-of-fold** prediction, from a model that never saw it, and all headline metrics are computed on those predictions. Folds are stratified on the binned target so every fold sees the full range of scores.

**Metrics** (both are reported everywhere):
* **RMSE**: how far predictions are from the human scores; squared, so large misses cost more. Lower is better.
* **Pearson r**: whether predictions rise and fall with the true scores, regardless of offset or scale. Higher is better.

Predictions are clipped to the valid range `[0, 5]`.

In [ ]:
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def pearson(y_true, y_pred):
    return float(pearsonr(y_true, y_pred)[0]) if np.std(y_pred) > 1e-12 else float("nan")

# Stratified folds on the binned target (same folds for every model => fair comparison)
strat_bins = pd.qcut(pd.Series(y_train).rank(method="first"), q=5, labels=False)
skf = StratifiedKFold(n_splits=CFG["n_folds"], shuffle=True, random_state=SEED)
folds = list(skf.split(X_tr, strat_bins))

MODELS = {
    "Ridge": lambda: make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(0, 4, 25))),
    "ExtraTrees": lambda: ExtraTreesRegressor(n_estimators=600, max_features=0.3, min_samples_leaf=2,
                                              n_jobs=-1, random_state=SEED),
    "RandomForest": lambda: RandomForestRegressor(n_estimators=500, max_features=0.3, min_samples_leaf=2,
                                                  n_jobs=-1, random_state=SEED),
    "HistGB": lambda: HistGradientBoostingRegressor(learning_rate=0.05, max_iter=300, max_leaf_nodes=15,
                                                    min_samples_leaf=15, l2_regularization=1.0, random_state=SEED),
}

def cv_predict(make_model, X, y):
    # Return one out-of-fold prediction per training row
    oof = np.zeros(len(y))
    for tr_idx, va_idx in folds:
        model = make_model()
        model.fit(X.iloc[tr_idx], y[tr_idx])
        oof[va_idx] = model.predict(X.iloc[va_idx])
    return np.clip(oof, 0, 5)

# Baseline: always predict the training mean (what "no skill" looks like)
oof_dummy = cv_predict(lambda: DummyRegressor(strategy="mean"), X_tr, y_train)
print(f"Baseline (predict the mean): OOF RMSE = {rmse(y_train, oof_dummy):.4f}")

### 7.1 All models on all fused features

In [ ]:
oof_preds, results = {}, []
for name, make in MODELS.items():
    t0 = time.time()
    oof_preds[name] = cv_predict(make, X_tr, y_train)
    results.append({"model": name, "OOF RMSE": rmse(y_train, oof_preds[name]),
                    "OOF Pearson": pearson(y_train, oof_preds[name]), "sec": round(time.time() - t0, 1)})
    print(f"{name:13s} RMSE = {results[-1]['OOF RMSE']:.4f} | Pearson = {results[-1]['OOF Pearson']:.4f}")

results_df = pd.DataFrame(results).set_index("model")
display(results_df.round(4))

### 7.2 Ablation: which view carries the signal?
The same Ridge and ExtraTrees models are trained on each feature group separately and on combinations. This answers the central question of the project: **does adding the transcript actually help over audio alone?**

In [ ]:
feature_sets = {
    "Acoustic only": blocks["acoustic"],
    "Linguistic stats only": blocks["linguistic"],
    "Embeddings only": blocks["embeddings"],
    "Linguistic + Embeddings (text only)": blocks["linguistic"] + blocks["embeddings"],
    "Acoustic + Linguistic": blocks["acoustic"] + blocks["linguistic"],
    "All (fusion)": list(X_tr.columns),
}
ablation = []
for fs_name, cols in feature_sets.items():
    if not cols:
        continue                      # block unavailable in this run
    for m_name in ["Ridge", "ExtraTrees"]:
        o = cv_predict(MODELS[m_name], X_tr[cols], y_train)
        ablation.append({"feature set": fs_name, "model": m_name, "n_features": len(cols),
                         "OOF RMSE": rmse(y_train, o), "OOF Pearson": pearson(y_train, o)})
ablation_df = pd.DataFrame(ablation)
display(ablation_df.pivot(index="feature set", columns="model", values=["OOF RMSE", "OOF Pearson"]).round(4))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sns.barplot(data=ablation_df, y="feature set", x="OOF RMSE", hue="model", ax=axes[0], palette=["#0B7A75", "#A85A0C"])
axes[0].axvline(rmse(y_train, oof_dummy), color="grey", ls="--", label="mean baseline")
axes[0].set(title="Ablation: OOF RMSE (lower is better)", ylabel="")
axes[0].legend(loc="lower right")
sns.barplot(data=ablation_df, y="feature set", x="OOF Pearson", hue="model", ax=axes[1], palette=["#0B7A75", "#A85A0C"])
axes[1].set(title="Ablation: OOF Pearson (higher is better)", ylabel="")
axes[1].legend_.remove()
plt.tight_layout(); plt.show()

## 8. Ensemble, final training and training RMSE

Different model families make different mistakes, so blending them usually beats any single one. Two blends are compared:

* **Equal-weight mean** of the four models.
* **NNLS stack**: non-negative weights fitted on the OOF predictions. Fitting weights on the same OOF predictions they are scored on would be optimistic, so the stack is scored with a *second* cross-validation (weights fitted on four folds, applied to the fifth).

Whichever blend has the lower honest OOF RMSE is used for the final model.

In [ ]:
names = list(MODELS)
P_oof = np.column_stack([oof_preds[n] for n in names])

def nnls_weights(P, y):
    w, _ = nnls(P, y)
    return w / w.sum() if w.sum() > 0 else np.full(P.shape[1], 1 / P.shape[1])

# Honest score of the stacked blend: weights learned on the training folds only
stack_oof = np.zeros(len(y_train))
for tr_idx, va_idx in folds:
    stack_oof[va_idx] = P_oof[va_idx] @ nnls_weights(P_oof[tr_idx], y_train[tr_idx])
stack_oof = np.clip(stack_oof, 0, 5)
mean_oof = np.clip(P_oof.mean(axis=1), 0, 5)

blend_scores = {
    "Equal-weight mean": (rmse(y_train, mean_oof), pearson(y_train, mean_oof)),
    "NNLS stack (CV-scored)": (rmse(y_train, stack_oof), pearson(y_train, stack_oof)),
}
display(pd.DataFrame(blend_scores, index=["OOF RMSE", "OOF Pearson"]).T.round(4))

if blend_scores["NNLS stack (CV-scored)"][0] < blend_scores["Equal-weight mean"][0]:
    final_w, oof_final, blend_name = nnls_weights(P_oof, y_train), stack_oof, "NNLS stack"
else:
    final_w, oof_final, blend_name = np.full(len(names), 1 / len(names)), mean_oof, "Equal-weight mean"
print(f"\nSelected blend: {blend_name}")
print("weights:", dict(zip(names, final_w.round(3))))

### 8.1 Final models on the full training set + compulsory training RMSE
Every model is refit on all 769 training clips. The **training RMSE** is reported next to the OOF RMSE: the gap between the two is the honest measure of overfitting.

In [ ]:
fitted, train_pred, test_pred = {}, {}, {}
for name, make in MODELS.items():
    fitted[name] = make().fit(X_tr, y_train)
    train_pred[name] = np.clip(fitted[name].predict(X_tr), 0, 5)
    test_pred[name] = np.clip(fitted[name].predict(X_te), 0, 5)

P_train = np.column_stack([train_pred[n] for n in names])
P_test = np.column_stack([test_pred[n] for n in names])
ens_train = np.clip(P_train @ final_w, 0, 5)
ens_test = np.clip(P_test @ final_w, 0, 5)

train_rmse_final = rmse(y_train, ens_train)
train_pearson_final = pearson(y_train, ens_train)
oof_rmse_final = rmse(y_train, oof_final)
oof_pearson_final = pearson(y_train, oof_final)

print("=" * 62)
print(f" TRAINING RMSE  (final ensemble, full train set): {train_rmse_final:.4f}")
print(f" TRAINING Pearson                               : {train_pearson_final:.4f}")
print(f" OOF RMSE       (5-fold, unseen-data estimate)  : {oof_rmse_final:.4f}")
print(f" OOF Pearson                                    : {oof_pearson_final:.4f}")
print("=" * 62)

final_table = pd.DataFrame({
    "Train RMSE": [rmse(y_train, train_pred[n]) for n in names] + [train_rmse_final],
    "Train Pearson": [pearson(y_train, train_pred[n]) for n in names] + [train_pearson_final],
    "OOF RMSE": [rmse(y_train, oof_preds[n]) for n in names] + [oof_rmse_final],
    "OOF Pearson": [pearson(y_train, oof_preds[n]) for n in names] + [oof_pearson_final],
}, index=names + [f"Ensemble ({blend_name})"])
final_table["RMSE gap (OOF - train)"] = final_table["OOF RMSE"] - final_table["Train RMSE"]
display(final_table.round(4))

> **How to read the gap.** Tree ensembles typically show a much smaller training RMSE than OOF RMSE because they can partly memorise the training set. The training RMSE is reported because the competition requires it, but **the OOF RMSE is the number to trust** as an estimate of performance on the 216 test clips.

## 9. Visualisations and error analysis

### 9.1 Model diagnostics (all based on OOF predictions)

In [ ]:
resid = y_train - oof_final
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Actual vs predicted
ax = axes[0, 0]
ax.scatter(y_train, oof_final, alpha=0.5, s=20, color="#0B7A75")
ax.plot([0, 5], [0, 5], "r--", label="perfect")
ax.set(title=f"Actual vs predicted (OOF)  r = {oof_pearson_final:.3f}", xlabel="actual score",
       ylabel="predicted score", xlim=(0, 5.2), ylim=(0, 5.2))
ax.legend()

# Residuals vs predicted
ax = axes[0, 1]
ax.scatter(oof_final, resid, alpha=0.5, s=20, color="#A85A0C")
ax.axhline(0, color="k", lw=1)
ax.set(title="Residuals vs predicted", xlabel="predicted score", ylabel="actual - predicted")

# Residual histogram
ax = axes[0, 2]
sns.histplot(resid, bins=25, kde=True, ax=ax, color="#0F1B33")
ax.axvline(0, color="r", ls="--")
ax.set(title=f"Residual distribution  (RMSE = {oof_rmse_final:.3f})", xlabel="actual - predicted")

# Prediction distribution: OOF vs test vs truth
ax = axes[1, 0]
sns.kdeplot(y_train, ax=ax, label="true (train)", color="k", fill=True, alpha=0.15)
sns.kdeplot(oof_final, ax=ax, label="OOF predictions", color="#0B7A75")
sns.kdeplot(ens_test, ax=ax, label="test predictions", color="#A85A0C")
ax.set(title="Prediction distribution (should show real spread)", xlabel="score")
ax.legend()

# Mean prediction per true-score bin (does it follow the diagonal?)
ax = axes[1, 1]
bins_true = np.round(y_train * 2) / 2
by_bin = pd.DataFrame({"true": bins_true, "pred": oof_final}).groupby("true")["pred"].agg(["mean", "std", "count"])
ax.errorbar(by_bin.index, by_bin["mean"], yerr=by_bin["std"], fmt="o-", color="#0B7A75", capsize=3)
ax.plot([0, 5], [0, 5], "r--")
ax.set(title="Mean OOF prediction per true score", xlabel="true score (rounded to 0.5)", ylabel="mean predicted")

# Fold-wise stability
ax = axes[1, 2]
fold_rmse = [rmse(y_train[va], oof_final[va]) for _, va in folds]
ax.bar([f"fold {i + 1}" for i in range(len(folds))], fold_rmse, color="#0F1B33")
ax.axhline(oof_rmse_final, color="r", ls="--", label=f"overall {oof_rmse_final:.3f}")
ax.set(title="RMSE per CV fold", ylabel="RMSE")
ax.legend()

plt.tight_layout(); plt.show()

if np.std(ens_test) < 0.1:
    print("WARNING: test predictions have almost no spread; something may be wrong upstream.")

**What to look for.** Points hugging the red diagonal in the first plot are good; a flatter-than-diagonal cloud means the model *shrinks towards the mean* (typical for regression on noisy human labels). Residuals should be centred on zero with no clear trend against the prediction. The test-prediction density should resemble the OOF density, not collapse to a spike.

### 9.2 Model and ensemble comparison

In [ ]:
cmp_df = final_table[["OOF RMSE", "OOF Pearson"]].copy()
cmp_df.loc["Mean baseline"] = [rmse(y_train, oof_dummy), np.nan]
cmp_df = cmp_df.sort_values("OOF RMSE")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].barh(cmp_df.index, cmp_df["OOF RMSE"], color="#0B7A75")
axes[0].set(title="OOF RMSE (lower is better)", xlabel="RMSE"); axes[0].invert_yaxis()
p_df = cmp_df.dropna().sort_values("OOF Pearson", ascending=False)
axes[1].barh(p_df.index, p_df["OOF Pearson"], color="#A85A0C")
axes[1].set(title="OOF Pearson (higher is better)", xlabel="Pearson r"); axes[1].invert_yaxis()
plt.tight_layout(); plt.show()

### 9.3 What drives the predictions?
Two complementary views: simple correlation of individual features with the human score, and the importance the ExtraTrees model assigns to each feature and feature group.

In [ ]:
# Top interpretable (non-embedding) features by |correlation| with the score
interp_cols = blocks["acoustic"] + blocks["linguistic"]
corr = X_tr[interp_cols].corrwith(pd.Series(y_train)).dropna()
top_corr = corr.reindex(corr.abs().sort_values(ascending=False).index).head(20)

imp = pd.Series(fitted["ExtraTrees"].feature_importances_, index=X_tr.columns).sort_values(ascending=False)
group_imp = imp.groupby(imp.index.map(block_of)).sum()

fig, axes = plt.subplots(1, 3, figsize=(20, 7))
colors = ["#0B7A75" if v > 0 else "#A85A0C" for v in top_corr.values[::-1]]
axes[0].barh(top_corr.index[::-1], top_corr.values[::-1], color=colors)
axes[0].set(title="Top 20 features by correlation with score", xlabel="Pearson r")

top_imp = imp.head(20)
axes[1].barh(top_imp.index[::-1], top_imp.values[::-1], color="#0F1B33")
axes[1].set(title="Top 20 ExtraTrees importances", xlabel="importance")

axes[2].bar(group_imp.index, group_imp.values, color=["#0B7A75", "#0F1B33", "#A85A0C"][:len(group_imp)])
axes[2].set(title="Importance by feature group", ylabel="summed importance")
plt.tight_layout(); plt.show()

### 9.4 Error analysis: where is the model wrong?
Metrics say *how much* the model is wrong; the largest errors say *why*. For each case, ask: was Whisper's transcript wrong or too clean? Is the accent hard for ASR? Is the response short but correct (over-estimated) or long and strong but hesitant (under-estimated)?

In [ ]:
err = all_df[is_train].reset_index(drop=True)[["filename", "duration", "transcript"]].copy()
err["actual"] = y_train
err["predicted"] = oof_final
err["abs_error"] = (err.actual - err.predicted).abs()
err["n_words"] = X_tr["ling_n_words"].values if "ling_n_words" in X_tr else np.nan
err["words_per_sec"] = X_tr["ling_words_per_sec"].values if "ling_words_per_sec" in X_tr else np.nan
err["pause_ratio"] = X_tr["ac_pause_ratio"].values if "ac_pause_ratio" in X_tr else np.nan

pd.set_option("display.max_colwidth", 160)
print("10 largest errors, model OVER-estimates grammar:")
display(err[err.predicted > err.actual].nlargest(10, "abs_error")[["filename", "actual", "predicted", "n_words", "words_per_sec", "pause_ratio", "transcript"]].round(2))
print("10 largest errors, model UNDER-estimates grammar:")
display(err[err.predicted < err.actual].nlargest(10, "abs_error")[["filename", "actual", "predicted", "n_words", "words_per_sec", "pause_ratio", "transcript"]].round(2))

In [ ]:
# Is the error systematic with respect to simple properties of the clip?
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for ax, col, label in zip(axes, ["n_words", "words_per_sec", "pause_ratio"],
                          ["words in transcript", "words per second", "pause ratio"]):
    ax.scatter(err[col], err.actual - err.predicted, alpha=0.4, s=16, color="#0F1B33")
    ax.axhline(0, color="r", ls="--")
    r = err[[col]].assign(res=err.actual - err.predicted).corr().iloc[0, 1]
    ax.set(title=f"Residual vs {label} (r = {r:.2f})", xlabel=label, ylabel="actual - predicted")
plt.tight_layout(); plt.show()

# Fluency is not grammar: compare how strongly fluency vs text features track the score
fluency_cols = [c for c in ["ling_words_per_sec", "ling_articulation_rate", "ac_pause_ratio", "ac_n_long_pauses", "ling_hesitation_ratio"] if c in X_tr]
print("Correlation of fluency-style features with the score:")
print(X_tr[fluency_cols].corrwith(pd.Series(y_train)).round(3).to_string())

**Grammar is not fluency.** A speaker can be fluent but grammatically weak (fast, confident, many errors) or grammatically strong but hesitant. A model built only on speech-rate and pause features would confuse the two. Combining the signal, the transcript statistics and the language embedding is what lets the model separate them, and the ablation in section 7.2 quantifies the gain.

## 10. Test predictions and submission file

In [ ]:
pred_by_file = dict(zip(all_df.loc[~is_train, "filename"], ens_test))

submission = sample_sub.copy()
mapped = submission[SUB_FILE_COL].map(pred_by_file)
if mapped.isna().any():                         # tolerate extension / path differences in the id column
    stem_map = {Path(str(k)).stem: v for k, v in pred_by_file.items()}
    mapped = mapped.fillna(submission[SUB_FILE_COL].map(lambda s: stem_map.get(Path(str(s)).stem)))
print("unmatched ids:", int(mapped.isna().sum()))
submission[SUB_LABEL_COL] = mapped.fillna(float(np.mean(y_train))).clip(0, 5).values

SUB_PATH = WORK_DIR / "submission.csv"
submission.to_csv(SUB_PATH, index=False)
print("saved:", SUB_PATH, "| rows:", len(submission))
display(submission.head())
print(submission[SUB_LABEL_COL].describe().round(3).to_string())

# Keep the fitted models for later reuse
try:
    import joblib
    joblib.dump({"models": fitted, "weights": dict(zip(names, final_w)), "features": list(X_tr.columns)},
                WORK_DIR / "grammar_models.joblib")
except Exception as exc:
    print("could not save models:", exc)

## 11. Report

### 11.1 Approach
Grammar scoring from speech is a **regression** problem with a small labelled set (769 clips). Grammar lives in *what is said*, so an audio-only classifier built on MFCCs would miss the main signal. The solution converts speech to text with Whisper, derives features from the transcript, and fuses them with acoustic features. Heavy representation learning is left to pretrained models; the small dataset is used only to learn the final mapping, which keeps overfitting under control.

### 11.2 Preprocessing
* Audio resampled to 16 kHz mono; leading / trailing silence trimmed (30 dB threshold).
* Audio inspected first: duration, sample rate, channels, waveform and Mel spectrograms, label distribution.
* Whisper transcripts cached; empty transcripts become empty strings (all text features then default to zero).
* Missing / infinite feature values imputed with **train-only** medians; zero-variance columns dropped.
* Predictions clipped to the valid range `[0, 5]`.

### 11.3 Pipeline architecture
1. **Acoustic branch**: MFCC (+ deltas), RMS, ZCR, spectral centroid / bandwidth / roll-off / contrast, chroma, pause statistics, each aggregated to mean / std / min / max.
2. **Language branch**: Whisper ASR transcript, then (a) hand-crafted linguistic features and (b) a frozen sentence-transformer embedding.
3. **Feature fusion**: concatenation of the three blocks, with cross-modal fluency features (words per second).
4. **Regressors**: Ridge, ExtraTrees, Random Forest, HistGradientBoosting.
5. **Selection**: 5-fold stratified OOF validation, equal-weight vs NNLS-stacked ensemble (stack scored with its own nested CV).
6. **Final model**: all regressors refit on the full training set, blended, clipped to `[0, 5]`.

In [ ]:
# Auto-generated results summary (numbers come from the cells above, nothing is typed in by hand)
best_single = final_table.iloc[:len(names)]["OOF RMSE"].idxmin()
def abl(fs_name):
    sel = ablation_df[(ablation_df["feature set"] == fs_name) & (ablation_df.model == "ExtraTrees")]["OOF RMSE"]
    return float(sel.iloc[0]) if len(sel) else float("nan")

text_only = abl("Linguistic + Embeddings (text only)")
audio_only = abl("Acoustic only")
fusion = abl("All (fusion)")

if not TEXT_AVAILABLE:
    display(Markdown("> **WARNING: this run used ACOUSTIC FEATURES ONLY** (no Whisper transcripts). "
                     "Expect weaker results; re-run with model access to get the full pipeline."))

display(Markdown(f"""
### 11.4 Evaluation results

| Quantity | RMSE | Pearson r |
|---|---|---|
| Mean-prediction baseline (OOF) | {rmse(y_train, oof_dummy):.4f} | n/a |
| Best single model, **{best_single}** (OOF) | {final_table.loc[best_single, 'OOF RMSE']:.4f} | {final_table.loc[best_single, 'OOF Pearson']:.4f} |
| **Final ensemble ({blend_name}), OOF (unseen-data estimate)** | **{oof_rmse_final:.4f}** | **{oof_pearson_final:.4f}** |
| **Final ensemble, TRAINING data (compulsory)** | **{train_rmse_final:.4f}** | {train_pearson_final:.4f} |

**Ablation (ExtraTrees, OOF RMSE):** acoustic only = {audio_only:.4f}, transcript-based features only = {text_only:.4f}, all fused = {fusion:.4f}.

**Generalisation gap:** training RMSE {train_rmse_final:.4f} vs OOF RMSE {oof_rmse_final:.4f} (gap {oof_rmse_final - train_rmse_final:+.4f}). The OOF figures are the reliable estimate for the 216 test clips; the training RMSE alone would be over-optimistic.
"""))

### 11.5 Interpretation
* The **ablation (section 7.2)** shows which view carries the signal and whether fusion beats any single view.
* The **actual-vs-predicted** and **per-score-bin** plots show whether the model separates weak, medium and strong speakers or mostly regresses to the mean.
* The **error analysis** shows the typical failure modes: transcripts that Whisper "cleaned up", short but correct answers, and long answers with hesitations.

### 11.6 Limitations
* **ASR smoothing:** Whisper may silently fix grammar mistakes, reducing the signal for the weakest speakers. Accents and noise can also cause recognition errors that look like grammar errors.
* **Small data:** 769 labels is little for text representations; frozen encoders and strong regularisation mitigate but do not remove variance.
* **Noisy labels:** human MOS scores are subjective, which limits the achievable RMSE and Pearson.
* **Hand-crafted features are proxies,** not a real grammar parser.

### 11.7 Possible improvements
* Add a grammar-aware signal: a rule-based checker (e.g. LanguageTool), a token-level error tagger, or language-model perplexity of the transcript.
* Use Whisper's own confidence scores and word-level timestamps for finer pause / disfluency features.
* Try a larger Whisper model, or compare several ASR systems; test-time augmentation of transcripts.
* Pool several frozen encoders (e.g. a speech encoder such as wav2vec 2.0 / WavLM in addition to the text encoder).
* Repeated CV with several seeds for more stable model selection, and calibration of the final scores to reduce shrinkage towards the mean.